# AI Career Project (Ordered)

This notebook trains a career prediction model from the dataset in `backend/data/`, saves artifacts, and runs predictions.

Optional: it can generate an explanation using Groq via `langchain-groq` **only if** `GROQ_API_KEY` is set in your environment.

In [ ]:
# If you're in Jupyter/Colab, run this once.
# In local venvs, prefer installing via pip/requirements.
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn langchain-groq


In [ ]:
import os
import numpy as np
import pandas as pd

import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


## 1) Load dataset

In [ ]:
BASE_DIR = os.path.abspath(os.getcwd())
DATA_PATH = os.path.join(BASE_DIR, "data", "post_ai_career_dataset_5000.csv")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
df.head()

## 2) Build the clean dataset used by the backend

The backend expects 3 numeric features: `AI_Skill`, `Coding_Skill`, `Communication_Skill`, and a target column `Career`.

In [ ]:
def ai_score(tools):
    tools = str(tools).lower()
    if "gpt" in tools or "ai" in tools:
        return 9
    if "aws" in tools or "cloud" in tools:
        return 7
    return 5

def coding_score(skills):
    skills = str(skills).lower()
    if "python" in skills:
        return 9
    if "sql" in skills:
        return 7
    return 5

def comm_score(domain):
    domain = str(domain).lower()
    if "business" in domain:
        return 9
    return 6

df_clean = df.copy()
df_clean["AI_Skill"] = df_clean["AI_Tool_Stack"].apply(ai_score)
df_clean["Coding_Skill"] = df_clean["Key_Skills"].apply(coding_score)
df_clean["Communication_Skill"] = df_clean["Primary_Domain"].apply(comm_score)
df_clean["Career"] = df_clean["Career_Role"].astype(str)

clean_df = df_clean[["AI_Skill", "Coding_Skill", "Communication_Skill", "Career"]].copy()
clean_df.head()

In [ ]:
CLEAN_PATH = os.path.join(BASE_DIR, "data", "clean_dataset.csv")
clean_df.to_csv(CLEAN_PATH, index=False)
print("Saved:", CLEAN_PATH)

## 3) Train model and save artifacts

In [ ]:
X = clean_df[["AI_Skill", "Coding_Skill", "Communication_Skill"]]
y = clean_df["Career"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
)
model.fit(X_train, y_train)

acc = model.score(X_test, y_test)
print("Accuracy:", round(acc, 4))

MODEL_PATH = os.path.join(BASE_DIR, "model.pkl")
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH)

## 4) Predict + optional Groq explanation

In [ ]:
model = joblib.load(MODEL_PATH)

def predict_career(ai_skill, coding_skill, communication_skill):
    ai_skill = float(ai_skill)
    coding_skill = float(coding_skill)
    communication_skill = float(communication_skill)
    career = model.predict([[ai_skill, coding_skill, communication_skill]])[0]
    return {
        "career": career,
        "ai_score": ai_skill,
        "coding_score": coding_skill,
        "communication_score": communication_skill,
    }

result = predict_career(5, 6, 7)
result

In [ ]:
def try_groq_explanation(raw_input: dict, result: dict):
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return None

    from langchain_groq import ChatGroq

    llm = ChatGroq(model_name="llama-3.1-8b-instant")
    prompt = (
        "You are a career advisor.\n"
        "Given the user scores and predicted career, explain why it fits and give 3 concrete next steps.\n\n"
        f"User scores: {raw_input}\n"
        f"Prediction: {result}\n"
    )
    return llm.invoke(prompt).content

explanation = try_groq_explanation(
    {"AI_Skill": result["ai_score"], "Coding_Skill": result["coding_score"], "Communication_Skill": result["communication_score"]},
    result,
)
explanation